## Extract the data required by RASSINE

For this notebook to work properly, a RAW directory is needed, containing S1D files (downloaded from DACE). The corresponding CCF files are also needed, the directory can be specified in the code.

A MASTER table will be generated, containing the minimum data required by RASSINE. The minimum columns needed in the master table are: mjd, fileroot, model, vrad, svrad, drift_used. The master table needs to be saved in a directory called DACE_TABLE.

To run RASSINE, run this command inside the directory that contains RAW and DACE_TABLE:

path/run_rassine.sh -c path/harpn.ini .

RASSINE needs to be installed in the current environment. Modify the paths so they match the locations of the files.

In [1]:
from astropy.io import fits

import os
import numpy as np
import pandas as pd
from datetime import datetime
import re  # find pattern in string

import tools as t

In [2]:
# define the path of the directory where dace data is saved
path_data = '../data/dace_sun/short_series/'
path_raw = path_data + 'RAW/'  # sub directory where the specific data is
path_ccf = path_data + 'CCF/'

In [3]:
# iterate over raw files
# save names of the files
files = []
for entry in os.scandir(path_raw):
    files.append(entry.name)

print(files)

['r.HARPN.2015-08-03T09_03_34.695_S1D_A.fits', 'r.HARPN.2015-08-03T11_18_49.195_S1D_A.fits', 'r.HARPN.2015-07-31T08_23_48.980_S1D_A.fits', 'r.HARPN.2015-07-30T13_45_50.967_S1D_A.fits', 'r.HARPN.2015-07-31T08_50_53.222_S1D_A.fits', 'r.HARPN.2015-08-03T15_16_55.692_S1D_A.fits', 'r.HARPN.2015-08-02T14_56_01.513_S1D_A.fits', 'r.HARPN.2015-07-29T10_50_01.027_S1D_A.fits', 'r.HARPN.2015-08-02T09_52_50.475_S1D_A.fits', 'r.HARPN.2015-08-03T10_30_12.749_S1D_A.fits', 'r.HARPN.2015-08-02T12_08_07.507_S1D_A.fits', 'r.HARPN.2015-08-02T14_18_08.829_S1D_A.fits', 'r.HARPN.2015-07-30T14_56_11.158_S1D_A.fits', 'r.HARPN.2015-07-29T10_39_10.408_S1D_A.fits', 'r.HARPN.2015-07-30T16_01_01.466_S1D_A.fits', 'r.HARPN.2015-08-02T13_13_07.404_S1D_A.fits', 'r.HARPN.2015-07-29T15_08_11.060_S1D_A.fits', 'r.HARPN.2015-07-29T10_33_45.100_S1D_A.fits', 'r.HARPN.2015-08-02T10_46_55.145_S1D_A.fits', 'r.HARPN.2015-07-30T16_11_50.990_S1D_A.fits', 'r.HARPN.2015-07-30T13_07_53.012_S1D_A.fits', 'r.HARPN.2015-08-02T10_14_28.670_

In [4]:
# save names and observation dates of the files
dates_files = []  # observation dates
files_list = []  # names
pattern = re.compile(r"\d{4}-\d{2}-\d{2}")
for entry in os.scandir(path_raw):  # iterate over raw files
    name = entry.name
    dates_files.append(t.extract_pattern(name, pattern))
    files_list.append(name)

dates_files = list(set(dates_files))  # remove repeated dates
dates_files = sorted(dates_files, key=lambda d: datetime.strptime(d, "%Y-%m-%d"))  # sort
print(dates_files)
print(f's1d files: {len(files_list)}')

['2015-07-29', '2015-07-30', '2015-07-31', '2015-08-02', '2015-08-03']
s1d files: 298


In [5]:
# make a list of the available CCF files
files_ccf_list = []
for d in dates_files:  # iterate over directories (which are named as observation dates)
    date_path = path_ccf + d + '/'
    try:
        files = os.scandir(date_path)  # files in directory
        files = list(files)
        files = [i.name for i in files]  # list of file names
        files_ccf_list.extend(files)  # save file names
    except Exception as e:
        print(f'No CCF data for {d}')
files_ccf_list = np.array(files_ccf_list)
print(f'ccf files: {len(files_ccf_list)}')

ccf files: 308


In [6]:
# check format of the files names
print(f'CCF files look like: {files_ccf_list[0]}')
print(f'S1D files look like: {files_list[0]}')

CCF files look like: r.HARPN.2015-07-29T11-33-18.498_CCF_A.fits
S1D files look like: r.HARPN.2015-08-03T09_03_34.695_S1D_A.fits


In [7]:
# keep only ccf files that have a corresponding s1d file
files_s1d = []
files_ccf = []

for i in files_ccf_list:  # iterate over ccf files
    # use the ccf file to reconstruct the name of the corresponding s1d file
    s1d_file = i.replace('_', '-')
    s1d_file = s1d_file.replace('-CCF-', '_S1D_')
    s1d_file = re.sub(r"(T\d{2})-(\d{2})-(\d{2})", r"\1_\2_\3", s1d_file)
    if s1d_file in files_list:  # if a corresponding s1d file exists
        files_s1d.append(s1d_file)  # save s1d file
        files_ccf.append(i)  # save ccf file

files_s1d = np.array(files_s1d)
files_ccf = np.array(files_ccf)
print(len(files_ccf), len(files_s1d))

298 298


In [8]:
# files_ccf and files_s1d have the same length and files at the same index are corresponding
print(files_ccf[0])
print(files_s1d[0])

r.HARPN.2015-07-29T11-33-18.498_CCF_A.fits
r.HARPN.2015-07-29T11_33_18.498_S1D_A.fits


In [9]:
# extract minimum data necessary to build the master table
# extract data from s1d files
mjd = []
fnames = []
models = []
for f in files_s1d:  # iterate over files
    path_file = path_raw + f
    hdul = fits.open(path_file)  # open file
    data = hdul[0].header
    mjd.append(data['MJD-OBS'])
    fnames.append('r.' + data['FILENAME'])
    models.append(data['HIERARCH TNG TEL TARG RADVEL'])

mjd = np.array(mjd)
fnames = np.char.replace(fnames, '.fits', '_S1D_A.fits')  # match the format required by RASSINE
fnames = np.array([re.sub(r"(T\d{2})-(\d{2})-(\d{2})", r"\1_\2_\3", i) for i in fnames])
models = np.array(models)

# extract data from ccf files
vrad = []
svrad = []
drift = []
bad_inds = []
for f in files_ccf:
    date_file = re.search(r"\d{4}-\d{2}-\d{2}", f).group()
    path_file = path_ccf + date_file + '/' + f
    try:
        hdul = fits.open(path_file)
        hdr0 = hdul[0].header
        vrad.append(hdr0['HIERARCH TNG QC CCF RV'])  # km/s
        svrad.append(hdr0['HIERARCH TNG QC CCF RV ERROR'])
        drift.append(0)
    except:  # account for problems with some files
        print(f'File {f} is not working')
        ind = np.where(files_ccf == f)
        bad_inds.append(ind[0][0])


# take out data corresponding to problematic files
mjd = np.array([x for i, x in enumerate(mjd) if i not in bad_inds])
fnames = np.array([x for i, x in enumerate(fnames) if i not in bad_inds])
models = np.array([x for i, x in enumerate(models) if i not in bad_inds])

vrad = np.array(vrad)
svrad = np.array(svrad)
drift = np.array(drift)

In [10]:
# create and save the master table
# path_out = path_data
path_table = '../tests/DACE_TABLE/Dace_extracted_table.csv'

dace_table = pd.DataFrame({
    'mjd': mjd,
    'fileroot': fnames,
    'model': models,
    'vrad': vrad,
    'svrad': svrad,
    'drift_used': drift
})

# sort by filename
dace_table = dace_table.sort_values('fileroot')

# reset row numbers after sorting
dace_table = dace_table.reset_index(drop=True)

# save
dace_table.to_csv(path_table, index=False)

In [11]:
# visualize the master table
df = pd.read_csv(path_table)
df

,mjd,fileroot,model,vrad,svrad,drift_used
0,57232.376181,r.HARPN.2015-07-29T09_01_42.883_S1D_A.fits,0.1,0.112070,0.000230,0
1,57232.379942,r.HARPN.2015-07-29T09_07_07.038_S1D_A.fits,0.1,0.111485,0.000229,0
2,57232.383704,r.HARPN.2015-07-29T09_12_32.347_S1D_A.fits,0.1,0.110983,0.000228,0
3,57232.387454,r.HARPN.2015-07-29T09_17_56.902_S1D_A.fits,0.1,0.111284,0.000227,0
4,57232.391227,r.HARPN.2015-07-29T09_23_22.210_S1D_A.fits,0.1,0.110922,0.000227,0
...,...,...,...,...,...,...
293,57237.621701,r.HARPN.2015-08-03T14_55_15.221_S1D_A.fits,0.1,0.111095,0.000222,0
294,57237.625475,r.HARPN.2015-08-03T15_00_41.298_S1D_A.fits,0.1,0.111495,0.000222,0
295,57237.629225,r.HARPN.2015-08-03T15_06_05.070_S1D_A.fits,0.1,0.111414,0.000222,0
296,57237.632998,r.HARPN.2015-08-03T15_11_31.150_S1D_A.fits,0.1,0.110874,0.000223,0
